<a href="https://colab.research.google.com/github/Prktriesitout/LangGraph/blob/main/LangGraphworkflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In Langraph, the **state** is a structured data container that holds all the information shared across the workflow
the state is the backbone of LangGraph workflow , It defines what data is available and how it evolves

For, a chatbot , the state might include :
the latest user input
the conversation history
intermediate result
Metadata (eg. user profile , time stamps )

How to define a state : Langgraph uses Python's TypedDIct , dataclass or pydantic.BaseModel to define ,

All nodes in the graph read from and write to the same state , ensuring consistency
Modularity: Nodes can be developed independently as long as they respect the state schema
We can inspect the state at any point to debug or "time travel" through the workflow

In [24]:
from typing import TypedDict, List, Optional
from IPython.display import Image, display
from langgraph.graph import StateGraph

class ChatState(TypedDict):
  input : str  # the latest message from the user
  messages : List[str] # A list of all messages exchanges so far
  tool_result: Optional[str] # Stores the output from any tool the chatbot might call


 # graph builder is the canvas use it to add notes ,
  #define transitions, and eventually compile the graph


A node in LangGraph is a function that performs a specific task in the workflow . It takes the current state as Input and returns an Updated states.
Node - represents the logic or operations that agent will perform

Ensure keep nodes modular and single_purpose .
Ensure they return a dictionary with updated state keys
Nodes can be sync and async

In [27]:
graph_builder = StateGraph(ChatState)

In [28]:
# adding a node

def greet_user(state: ChatState):
  user_input = state["input"]
  response = f"Hello !  You said: {user_input}"
  state["messages"].append(response)
  return {"messages": state["messages"]}

graph_builder.add_node("greet",greet_user)


Add an Entry Point
The entry point is the starting node of your graph. It tells LangGraph where to begin execution when the workflow

Note :  You can only have one entry point ,

Choose a node that makes sense as the first step in your logic

In [29]:
graph_builder.set_entry_point("greet")

Add an Exit Point

An exit point defines where the graph should terminate .

In [30]:
graph_builder.set_finish_point("end")

also define the end node .

In [31]:
def end_conversation(state):
  return {}
graph_builder.add_node("end", end_conversation)

In [32]:
graph_builder.add_edge(START, "greet")
graph_builder.add_edge("greet", "end")
graph_builder.add_edge("end", END)


can have multiple exit points if needed
Use exit points to gracefully terminate the workflow after completing the task

summary
1. Add Node :
2. Add Entry Point:
3. Add Exit Point:

Compile the graph

In [33]:
from IPython.display import Image, display
from langgraph.graph import END, START

app = graph_builder.compile()
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	greet(greet)
	end(end)
	__end__([<p>__end__</p>]):::last
	__start__ --> greet;
	greet --> end;
	end --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



.invoke() -> gives the output without intermediate steps and
.stream() -> observe step by step execution

In [34]:

result = app.invoke({"input": "Book a flight from NYC to LA", "messages": []})
print(result)

{'input': 'Book a flight from NYC to LA', 'messages': ['Hello !  You said: Book a flight from NYC to LA']}


In [36]:
for step in app.stream({"input": "Book a flight from NYC to LA","messages": []}):
    print("Step:", step)

Step: {'greet': {'messages': ['Hello !  You said: Book a flight from NYC to LA']}}
Step: {'end': None}


from decimal import InvalidOperation
features of Langgraph :
**StateGraph :** Main class for creating a graph
**add_node()** : Adds a node to the graph
**add_edge() :** Adds a direct edge between two node
**add_constitutional_edges () :** Adds conditinal edges based on a routing function
**set_entry_point() :** Sets the entry point of the graph
**set_finish_point() :** Sets the finish point of the graph
**compile() :** Compiles the graph into a runnable graph
**invoke() :** Invokes the graph with the given input
**stream() :** Streams the graph with the given input

In [ ]:
# This code demonstrates how the data flows through nodes and how messages with metadata are stored #and updated in the state We’ll build a two-node workflow:
#Node 1 – Summarizer: Takes user input → creates a summary → adds an annotated AIMessage.
#Node 2 – Validator: Checks if the summary length is acceptable → appends another annotated message.
#Finally, the workflow will return the full conversation state with all annotated messages.
# -----------------------------
# Imports
# -----------------------------
from typing import TypedDict, List
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
# -----------------------------
#  Step 1: Define the State
# -----------------------------
class SummarizationState(TypedDict):
    user_input: str
    summary: str
    messages: List  # This will hold annotated messages
# -----------------------------
# 🧩 Step 2: Define the Nodes
# -----------------------------
# Node 1: Summarizer
def summarizer_node(state: SummarizationState):
    text = state["user_input"]
    summary = text[:60] + "..."  # simple placeholder summary
    # Annotate the AI message
    ai_msg = AIMessage(
        content=f"Summary: {summary}",
        metadata={"node": "summarizer", "type": "summary"}
    )
    # Update the state
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["summary"] = summary
    state["messages"] = messages
    return state
# Node 2: Validator
def validator_node(state: SummarizationState):
    summary = state["summary"]
    valid = len(summary) < 80
    validation_result = "Valid summary" if valid else "Too long summary"
    ai_msg = AIMessage(
        content=validation_result,
        metadata={"node": "validator", "type": "validation"}
    )
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["messages"] = messages
    return state
# -----------------------------
#  Step 3: Create the Graph
# -----------------------------
graph = StateGraph(SummarizationState)
graph.add_node("summarizer", summarizer_node)
graph.add_node("validator", validator_node)
graph.add_edge(START, "summarizer")
graph.add_edge("summarizer", "validator")
graph.add_edge("validator", END)
# Compile the graph
app = graph.compile()
# -----------------------------
# Step 4: Run the Graph
# -----------------------------
input_text = "LangGraph simplifies building stateful AI agents using Python graphs."
initial_state = {
    "user_input": input_text,
    "summary": "",
    "messages": [HumanMessage(content=input_text)]
}
# Run the workflow
final_state = app.invoke(initial_state)
# -----------------------------
#  Step 5: Observe Results
# -----------------------------
print("=== Final State ===")
print(final_state)
print("\n=== Annotated Messages ===")
for msg in final_state["messages"]:
    meta = getattr(msg, "metadata", {})  # Use empty dict if not present
    print(f"{msg.type.upper()} -> {msg.content} | metadata={meta}")


This is a demo code (will not work)

In [ ]:
# -----------------------------
# Step 1: Define the State
# -----------------------------
class AgentState(TypedDict):
    user_input: str
    summary: str
    messages: List  # Annotated message history
    decision: str   # "summarize", "validate", or "end"
# -----------------------------
# Step 2: Define Nodes
# -----------------------------
# Node 1 - LLM decides what to do next
def controller_node(state: AgentState):
    """Ask the LLM: what should be done next?"""
    user_input = state["user_input"]
    messages = state.get("messages", [])
    prompt = f"""
You are an AI workflow controller.
User said: {user_input}
Decide the next step in the workflow.
Options:
1. summarize - if user input is long text
2. validate - if there is already a summary
3. end - if summary is valid or no action needed
Answer with ONLY one word: summarize, validate, or end.
"""
    decision = llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
    state["decision"] = decision
    return state
# Node 3 - Summarization
def summarizer_node(state: AgentState):
    text = state["user_input"]
    summary = llm.invoke([
        HumanMessage(content=f"Summarize this text briefly:\n\n{text}")
    ]).content.strip()
    state["summary"] = summary
    ai_msg = AIMessage(
        content=f"Summary: {summary}",
        metadata={"node": "summarizer"}
    )
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["messages"] = messages
    return state
# Node 4 - Validator
def validator_node(state: AgentState):
    summary = state["summary"]
    validation = llm.invoke([
        HumanMessage(content=f"Is the following summary under 80 words? Reply only with Yes or No.\n\n{summary}")
    ]).content.strip()
    ai_msg = AIMessage(
        content=f"Validation result: {validation}",
        metadata={"node": "validator"}
    )
    messages = state.get("messages", [])
    messages.append(ai_msg)
    state["messages"] = messages
    return state
# -----------------------------
# Step 5: Build the Graph
# -----------------------------
graph = StateGraph(AgentState)
graph.add_node("controller", controller_node)
graph.add_node("summarizer", summarizer_node)
graph.add_node("validator", validator_node)
# Start → Controller
graph.add_edge(START, "controller")
# Conditional transitions based on decision
graph.add_conditional_edges(
    "controller",
    lambda state: state["decision"],
    {
        "summarize": "summarizer",
        "validate": "validator",
        "end": END,
    }
)
# After summarization → validation
graph.add_edge("summarizer", "validator")
# After validation → End
graph.add_edge("validator", END)
# Compile
app = graph.compile()
# -----------------------------
# Step 7: Run the Agent
# -----------------------------
input_text = """
LangGraph is a Python framework that helps developers build complex,
stateful, multi-step AI agents using a directed graph of nodes.
"""
initial_state = {
    "user_input": input_text,
    "summary": "",
    "messages": [HumanMessage(content=input_text)],
    "decision": ""
}
# Run the dynamic graph
final_state = app.invoke(initial_state)
# -----------------------------
# Step 7: Print Results
# -----------------------------
print("=== Final State ===")
print(final_state)
print("\n=== Annotated Messages ===")
for msg in final_state["messages"]:
    meta = getattr(msg, "metadata", {})  # Use empty dict if not present
    print(f"{msg.type.upper()} -> {msg.content} | metadata={meta}")


Memory / Persistence : Enable agents to retain and recall information across steps, sessions , or tasks, supporting contextual continuity and personalization

Tools : Extend agent capabilities by integrating APIs, utilities and domainspecific functions , TOls empower nodes to perform external actions like API calls , calcualtions or database queries

Human in the loop : Introduces checkpoints for human Oversight , feedback or decision making ensuring safety , alignment and adaptability

State Customization : Tailor the agent's internal state representation to suit complex worflows, multi-agent interactions or domain specific knowledge

Time Travel : Navigate and manipulate historical states to support debugging , auditing and iterative refining of agent behaviour

In [39]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import InMemorySaver

# --- 1. Define the state schema ---
class AgentState(TypedDict, total=False):
    a: Optional[str]
    b: Optional[str]

# --- 2. Define nodes ---
def node_a(state: AgentState) -> AgentState:
    state["a"] = "Hello"
    return state

def node_b(state: AgentState) -> AgentState:
    state["b"] = state["a"] + " World!"
    return state

# --- 3. Build graph ---
builder = StateGraph(AgentState)

builder.add_node("A", node_a)
builder.add_node("B", node_b)

builder.set_entry_point("A")
builder.add_edge("A", "B")

# --- 4. Compile with a checkpoint saver ---
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "thread_1"}}
final_state = graph.invoke({}, config=config)
print(final_state)


{'a': 'Hello', 'b': 'Hello World!'}


time Travel : It is important as it is the ability to revisit and re-execute a graph from a previously saved checkpoint , this is made possible by its persistence layer , which stores the state of the graph at each super - step during execution

How Time Travel Works
LangGraph saves checkpoints tied to a thread ID. Each checkpoint captures:

The current state values

Execution metadata

Configuration

Next nodes to execute

config = {
"configurable": {
"thread_id": "user_123",
"checkpoint_id": "abc123"
}
}
graph.invoke(None, config=config)


In [40]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import InMemorySaver

# --- State Schema ---
class AgentState(TypedDict, total=False):
    name: Optional[str]
    greeting: Optional[str]
    message: Optional[str]

# --- Nodes ---
def greet_node(state: AgentState) -> AgentState:
    state["greeting"] = "Hello"
    return state

def personalize_node(state: AgentState) -> AgentState:
    state["message"] = f"{state['greeting']}, {state.get('name', 'Guest')}!"
    return state

# --- Build Graph ---
builder = StateGraph(AgentState)
builder.add_node("Greet", greet_node)
builder.add_node("Personalize", personalize_node)
builder.set_entry_point("Greet")
builder.add_edge("Greet", "Personalize")

# --- Compile Graph ---
graph = builder.compile(checkpointer=InMemorySaver())

# --- Config ---
config = {"configurable": {"thread_id": "session_1"}}

# --- Run Graph ---
initial_state = {"name": "ABC"}
final_state = graph.invoke(initial_state, config=config)
print("Final State:", final_state)
############################################################
# --- Get History ---
history = list(graph.get_state_history(config=config))
print("\nCheckpoints:")
for i, entry in enumerate(history):
    if isinstance(entry, tuple):
        checkpoint_id = entry[0]
        data = entry[1]
        values = data.get("values") if isinstance(data, dict) else data
        print(f"{i}: ID={checkpoint_id}, State={values}")
    elif isinstance(entry, dict):
        print(f"{i}: ID={entry.get('checkpoint_id')}, State={entry.get('values')}")

# --- Update State ---

graph.update_state( config, values={"name": "User_1"})

# Time travel (just re-invoke from start)
time_travel_config = {"configurable": {"thread_id": "session_1"}}
new_state = graph.invoke(None, config=time_travel_config)
print("Time Travelled State:", new_state)


Final State: {'name': 'ABC', 'greeting': 'Hello', 'message': 'Hello, ABC!'}

Checkpoints:
0: ID={'name': 'ABC', 'greeting': 'Hello', 'message': 'Hello, ABC!'}, State=()
1: ID={'name': 'ABC', 'greeting': 'Hello'}, State=('Personalize',)
2: ID={'name': 'ABC'}, State=('Greet',)
3: ID={}, State=('__start__',)
Time Travelled State: {'name': 'User_1', 'greeting': 'Hello', 'message': 'Hello, ABC!'}


Tools in LangGraph are external functions or capabilities that nodes can invoke during execution.

LangGraph supports multiple patterns for tool integration
Single Tool Node : A node dedicated to invoking a specific tool
Tool Router Node : A dynamic node that selects among multiple tools based on user intent or agent reasoning
Tool + Memory Node : Combines tool invocation with memory updates , enabling context aware actions .

In [42]:
from langchain.agents import Tool
search_tool = Tool(    name="WebSearch",    func=search_function,    description="Useful for answering questions about current events or facts.")


ERROR: unknown command "langchain_core"


Interrupt :
Enabling Human-in-the-Loop Workflows : Interrups in LangGraph provide a powerful mechanism to pause graph execution at runtime

it works in two state :
Triggering an Interrupt
Within any node logic, invoke the interrupt() function. This function:

Accepts any JSON-serializable payload (e.g., a message, a data structure, or a prompt).

Immediately halts execution and returns control to the caller.

Surfaces the payload in the __interrupt__ field of the response.

Resuming Execution
To resume the graph:

Re-invoke the graph using the same thread_id and provide the necessary input.

The resumed execution continues from the point of interruption.

The return value of the original interrupt() call becomes the input provided during resumption.

You are tasked with building a simple approval system using LangGraph where:

A user initiates a request to perform a sensitive action (e.g., "Transfer $500").

The system pauses execution and prompts for human approval using the interrupt() function.

Based on the human's response (True for approval, False for rejection), the workflow either:

Proceeds to execute the action and marks the status as "approved", or

Cancels the action and marks the status as "rejected"

Requirements:

Define a TypedDict-based state to hold:

action_details: A string describing the action.

status: A string indicating the current status ("pending", "approved", or "rejected").

Implement the following nodes:

approval_node: Uses interrupt() to pause and request approval.

proceed_node: Updates the status to "approved".

cancel_node: Updates the status to "rejected".

Use a MemorySaver checkpointer for state persistence.

Use a thread_id to manage execution state and resume the graph after interruption.

Demonstrate both the initial invocation (which pauses) and the resumption with a decision

In [45]:
from typing import Literal, Optional, TypedDict
from typing_extensions import Annotated
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
# --- Define State Schema ---
class ApprovalState(TypedDict):
    action_details: str
    status: Annotated[Optional[Literal["pending", "approved", "rejected"]], "replace"]
# --- Nodes ---
def approval_node(state: ApprovalState) -> Command:
    """Ask for approval and pause for user input."""
    decision = interrupt({
        "question": "Approve this action?",
        "details": state["action_details"],
        "options": ["Approve", "Reject"]
    })
    # Route dynamically based on user input at resume
    if decision.lower() == "approve":
        return Command(goto="proceed")
    else:
        return Command(goto="cancel")
def proceed_node(state: ApprovalState) -> ApprovalState:
    print("Proceeding with approval...")
    return {"status": "approved"}
def cancel_node(state: ApprovalState) -> ApprovalState:
    print("Action rejected.")
    return {"status": "rejected"}
# --- Build Graph ---
builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)
# --- Compile ---
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)
# --- Run Graph (Interactive Flow) ---
config = {"configurable": {"thread_id": "approval-123"}}
# Step 1: Trigger interrupt
initial = graph.invoke(
    {"action_details": "Transfer $500", "status": "pending"},
    config=config,
)
interrupts = initial.get("__interrupt__")
if interrupts:
    question = interrupts[0].value["question"]
    details = interrupts[0].value["details"]
    options = interrupts[0].value.get("options", [])
    print("\n--- Approval Request ---")
    print(f"Action: {details}")
    print(f"{question}")
    print(f"Options: {options}")
    # Take user decision
    user_input = input("Enter your decision (Approve/Reject): ").strip()
    # Step 2: Resume with user input
    resumed = graph.invoke(Command(resume=user_input), config=config)
    print("\n--- Final Decision ---")
    print(f"User chose: {user_input}")
    print(f"Final Status: {resumed['status']}")
else:
    print("No interrupt found process completed.")



--- Approval Request ---
Action: Transfer $500
Approve this action?
Options: ['Approve', 'Reject']
Enter your decision (Approve/Reject): yes
Action rejected.

--- Final Decision ---
User chose: yes
Final Status: rejected


In [46]:
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
# --- Define the state ---
class ReviewState(TypedDict):
    topic: str
    generated_text: str
    final_text: str
# --- Step 1: AI Draft Generation (Simulated) ---
def generate_draft_node(state: ReviewState):
    topic = state["topic"]
    draft = (
        f"This is an AI-generated draft about {topic}. "
        f"It provides a quick overview that may need improvement for tone and depth."
    )
    print(f"\nGenerated Draft:\n{draft}\n")
    return {"generated_text": draft}
# --- Step 2: Human Review / Edit Node (Interrupt) ---
def review_node(state: ReviewState):
    print(" Waiting for human review...")
    updated_text = interrupt({
        "instruction": "Please review and edit the AI-generated content below.",
        "content": state["generated_text"],
    })
    return {"final_text": updated_text}
# --- Step 3: Finalization Node ---
def finalize_node(state: ReviewState):
    print("\n Final content has been reviewed and approved!")
    return state
# --- Build the Graph ---
builder = StateGraph(ReviewState)
builder.add_node("generate_draft", generate_draft_node)
builder.add_node("review", review_node)
builder.add_node("finalize", finalize_node)
builder.add_edge(START, "generate_draft")
builder.add_edge("generate_draft", "review")
builder.add_edge("review", "finalize")
builder.add_edge("finalize", END)
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)
config = {"configurable": {"thread_id": "review-session-001"}}
# --- Step 1: Start the Workflow ---
initial_state = {"topic": "Responsible AI development", "generated_text": "", "final_text": ""}
result = graph.invoke(initial_state, config=config)
# --- Step 2: Handle Interrupt ---
interrupts = result.get("__interrupt__")
if interrupts:
    info = interrupts[0].value
    print("\n--- Human Review Required ---")
    print(f"{info['instruction']}")
    print(f"\nAI Draft:\n{info['content']}\n")
    # --- Get human input ---
    print("Enter your reviewed/edited text below (press Enter twice to submit):")
    human_edit = []
    while True:
        line = input()
        if not line:
            break
        human_edit.append(line)
    human_edit_text = "\n".join(human_edit).strip()
    if not human_edit_text:
        human_edit_text = info["content"]  # fallback to original if no edit made
    # --- Step 3: Resume Graph with Human Edit ---
    final = graph.invoke(Command(resume=human_edit_text), config=config)
    print("\n--- Final Reviewed Content ---")
    print(final["final_text"])
else:
    print("No human input required. Workflow completed automatically.")





Generated Draft:
This is an AI-generated draft about Responsible AI development. It provides a quick overview that may need improvement for tone and depth.

 Waiting for human review...

--- Human Review Required ---
Please review and edit the AI-generated content below.

AI Draft:
This is an AI-generated draft about Responsible AI development. It provides a quick overview that may need improvement for tone and depth.

Enter your reviewed/edited text below (press Enter twice to submit):
so this is fine
so this is fone
so this is fine

 Waiting for human review...

 Final content has been reviewed and approved!

--- Final Reviewed Content ---
so this is fine
so this is fone
so this is fine


In [47]:
from langchain.tools import tool
from langgraph.types import interrupt
@tool
def send_email(to: str, subject: str, body: str):
    """Send an email to a recipient."""
    # Pause before sending; payload surfaces in result["__interrupt__"]
    response = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "Approve sending this email?"
    })
    if response.get("action") == "approve":
        # Resume value can override inputs before executing
        final_to = response.get("to", to)
        final_subject = response.get("subject", subject)
        final_body = response.get("body", body)
        return f"Email sent to {final_to} with subject '{final_subject}'"
    return "Email cancelled by user"


Now , we will be talking about Memory

there are three levels of memory :-   
Task Level memory :    
Conversation-level memory :    
long-term contextual memory :     

Short-term memory, also known as thread-level persistence, allows agents to maintain context across multiple turns in a conversation. This is essential for building coherent, stateful interactions.

To enable short-term memory:
Use a checkpointer (e.g., MemorySaver, SqliteSaver) to persist the graph state.
Assign a unique thread_id in the graph configuration to identify and resume the same conversation thread across invocations.
Ensure your state includes a messages field (or similar) to accumulate and track the dialogue history.

Common Strategies for Managing Short-Term Memory:
Message Trimming
Remove the oldest or least relevant messages from the history before invoking the LLM to stay within the context window.

State Pruning
Permanently delete older messages from the LangGraph state to reduce memory footprint.

Summarization
Summarize earlier parts of the conversation and replace detailed messages with concise summaries to preserve key context.

Checkpoint Management
Use checkpoints to store and retrieve message history efficiently, enabling rollback or selective restoration of state.

Custom Memory Policies
Implement domain-specific strategies such as filtering messages by type, importance, or recency to optimize memory usage.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, AIMessage
# Define a simple model node
def call_model(state: MessagesState):
    last_user_message = state["messages"][-1].content
    response = AIMessage(content=f"You said: {last_user_message}")
    return {"messages": state["messages"] + [response]}
# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.set_entry_point("call_model")
builder.add_edge("call_model", END)
# Add short-term memory using in-memory checkpointer
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)
# Invoke the graph with a thread ID
graph.invoke(
    {"messages": [HumanMessage(content="Hi, I'm ABC")]},
    {"configurable": {"thread_id": "thread-1"}}
)


2. Long-term Memory
Long-term memory enables your agent to persist user-specific or application-specific data across multiple sessions or conversations. This allows the agent to recall past interactions, preferences, or contextual information, enabling more personalized and context-aware experiences over time.

To implement long-term memory:

Integrate a durable checkpointer (e.g., SqliteSaver, RedisSaver, or a custom backend).

Design your state schema to include fields for storing long-lived data (e.g., user profile, preferences, historical interactions).

Use a consistent thread_id or a user identifier to associate memory with a specific user or session.

In [48]:
from langgraph.store.memory import InMemoryStore
# Define a dummy embedding function (replace with real embeddings in production)
def embed(texts: list[str]) -> list[list[float]]:
    return [[1.0, 2.0] * len(texts)]
# Create the store with semantic search enabled
store = InMemoryStore(index={"embed": embed, "dims": 2})
# Define namespace and key
namespace = ("user_123", "preferences")
memory_key = "user_profile"
# Store long-term memory
store.put(
    namespace,
    memory_key,
    {
        "name": "USER-2",
        "preferences": ["short responses", "English", "Python"]
    }
)
# Retrieve memory
retrieved = store.get(namespace, memory_key)
print("Retrieved Memory:", retrieved)
# Semantic search example
results = store.search(namespace, query="language preference", limit=1)
print("Semantic Search Result:", results)


Retrieved Memory: Item(namespace=['user_123', 'preferences'], key='user_profile', value={'name': 'USER-2', 'preferences': ['short responses', 'English', 'Python']}, created_at='2026-11-17T17:24:23.343033+00:00', updated_at='2026-11-17T17:24:23.343037+00:00')
Semantic Search Result: [Item(namespace=['user_123', 'preferences'], key='user_profile', value={'name': 'USER-2', 'preferences': ['short responses', 'English', 'Python']}, created_at='2026-11-17T17:24:23.343033+00:00', updated_at='2026-11-17T17:24:23.343037+00:00', score=0.9999999999999998)]


Graph API

The Graph API models workflows as graphs with:

State: A TypedDict, dataclass, or Pydantic model that defines the schema.
Nodes: Functions that operate on the state.
Edges: Define transitions between nodes.
Components
1. State

Central to the graph.
Can include reducers to manage how updates are applied.
Supports short-term memory and long-term memory.
2. Nodes

Accept the current state and return updates.
Can be synchronous or asynchronous.
Can access runtime context and configuration.
3. Edges

Define control flow.
Types:
Normal Edges: Fixed transitions.
Conditional Edges: Dynamic routing based on state.
Send: For map-reduce patterns.
Command: Combines state update and routing.
4. Compilation

Graphs must be compiled before execution using .compile().
Compilation checks for structural correctness and sets runtime options.
5. Execution

Use .invoke(), .ainvoke(), .stream(), or .astream() to run the graph.
6. Visualization

Graphs can be visualized, aiding debugging and documentation.

The Functional API allows you to build workflows using standard Python functions and decorators, without explicitly modeling a graph.

 Components:
1. @entrypoint

Marks the start of a workflow.

Manages execution, checkpointing, and human-in-the-loop (HITL).

Accepts a single argument (use a dictionary for multiple inputs).

Returns a Pregel instance for execution.



2. @task

Defines a unit of work (e.g., API call, computation).

Supports asynchronous execution and checkpointing.

Returns a future-like object that can be awaited or resolved.



3. Interrupts

Use interrupt() to pause execution for human input.

Resume with Command(resume=...).



4. Short-term Memory   

Use previous parameter to access prior state in the same thread.



5. Serialization   

Inputs/outputs must be JSON-serializable for checkpointing.



6. Execution

Similar to Graph API: .invoke(), .ainvoke(), .stream(), .astream().

Architecture Design

A multi-agent AI pipeline in LangGraph can be structured as follows:

Parent Graph (Coordinator):
Responsible for orchestrating the overall workflow.
Invokes agent subgraphs and manages the shared state.
Streams outputs from subgraphs to clients or dashboards.
Agent Subgraphs:
Each subgraph represents an individual agent.
Encapsulates its nodes for a specific functionality (e.g., summarization, question answering, data validation).
Can stream incremental outputs to the parent graph or directly to clients.
Shared State / Messaging Layer:
Facilitates communication between agents.
Ensures consistent updates, conflict resolution, and real-time monitoring